<a href="https://colab.research.google.com/github/raimund-buehler/visualizing-the-brain/blob/main/notebooks/day_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day One: From Brain Images to a First fMRI Comparison

Today we will use Python to open brain images, inspect the numbers inside them, and ask a simple question of real fMRI data.

By the end, you will be able to:

- describe voxels, slices, volumes, and 4D fMRI data,
- inspect an MRI image as an array of numbers,
- navigate through the brain with different plots,
- explain why an fMRI image needs to be compared with something,
- describe the basic idea of a GLM and a contrast, and
- calculate a first-level contrast for one participant.

## What is a Jupyter notebook?

A Jupyter notebook is a page where text, code, pictures, and results can live together.

This notebook has two kinds of boxes, called **cells**:

- **Text cells**, like this one, explain what is happening.
- **Code cells** contain Python instructions that the computer can run.

To run a code cell, click the play button on the left. You can also click inside the cell and press **Shift + Enter**.

Run the cells from top to bottom. A later cell may need a variable that was created in an earlier cell.

## Setup: press play here first

The next cell installs and imports our tools. It also loads a standard anatomical brain template.

You do not need to understand every setup line yet. Run it once and wait for **Setup complete**.

In [ ]:
# @title Setup: install tools and load an anatomical template
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nilearn", "nibabel", "matplotlib", "numpy", "pandas"
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets, image, plotting
from nilearn.glm.first_level import (
    FirstLevelModel,
    make_first_level_design_matrix,
    spm_hrf,
)

%matplotlib inline

# This is an average anatomical brain in a shared reference space.
anat_img = datasets.load_mni152_template(resolution=2)

print("Setup complete.")
print("Anatomical image shape:", anat_img.shape)

## MRI data: anatomy and function

MRI scanners can create different kinds of data. We will use two today:

- An **anatomical MRI** is a detailed 3D picture of brain structure. Each voxel has one brightness value.
- **fMRI** records many 3D brain volumes, one after another. Each voxel therefore has a series of values over time.

A scanner collects the brain in **slices**. A stack of slices makes one 3D **volume**. In fMRI, many volumes make a 4D data set.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/mri_fmri_build_up.png" alt="MRI and fMRI data built from voxels, slices, volumes, and time" width="850">

The highlighted green cube is one **voxel**. A voxel is like a 3D pixel.

## A brain image is an array of numbers

A computer does not begin with a picture of a brain. It stores a grid of numbers called an **array**. Plotting software turns those numbers into brightness or color.

For a 3D anatomical image, the array has three dimensions:

- the first position moves through the **x** direction,
- the second moves through the **y** direction, and
- the third moves through the **z** direction.

The image's `shape` tells us how many voxel positions exist along each dimension.

### Mini task: turn the brain image into an array

The setup cell loaded an anatomical brain image and saved it in a variable called `anat_img`.

Your task is to complete the next code cell so that Python can inspect the image as numbers. Work with a neighbor first. If you get stuck, use the emergency solution below.

**Hint:** `get_fdata()` means “get the image data as floating-point numbers.”


In [ ]:
# Mini task: complete the two blanks.
# 1. Save the numerical data in a variable called anat_data.

anat_data = ________.get_fdata()

print("Python object:", type(________))
print("Array type:", type(anat_data))
print("Array shape:", anat_data.shape)
print("Number of dimensions:", anat_data.ndim)


<details>
<summary>Emergency solution: click only if your group is stuck</summary>

```python
anat_data = anat_img.get_fdata()

print("Python object:", type(anat_img))
print("Array type:", type(anat_data))
print("Array shape:", anat_data.shape)
print("Number of dimensions:", anat_data.ndim)
```

</details>


### Reading one voxel

We can use three array positions to ask for one voxel value. 

The code below finds the middle array position automatically. The three numbers are an **array index**. They identify a stored voxel.

In [ ]:
# Find the middle index along x, y, and z.
middle_x = anat_data.shape[0] // 2
middle_y = anat_data.shape[1] // 2
middle_z = anat_data.shape[2] // 2

middle_value = anat_data[middle_x, middle_y, middle_z]

print("Middle array index:", (middle_x, middle_y, middle_z))
print("Value stored there:", middle_value)

An array can also give us a small block of nearby voxels. The colon in `a:b` means: start at `a` and stop just before `b`.

First, we will select a `3 × 3 × 3` block around the middle voxel. Then we will plot it as a small 3D object.


In [ ]:
# Select a 3 × 3 × 3 block around the middle.
small_block = anat_data[
    middle_x - 1:middle_x + 2,
    middle_y - 1:middle_y + 2,
    middle_z - 1:middle_z + 2,
]

print("Block shape:", small_block.shape)
print(small_block)


### Turn the numbers into a 3D object

The next cell draws the selected `3 × 3 × 3` block as 27 cubes.

Each cube is one voxel. Its color comes from the number stored at that position: darker purple means a lower value and brighter yellow means a higher value.


In [ ]:
# Scale the values between 0 and 1 so they can become colors.
block_range = np.ptp(small_block)
scaled_values = (small_block - small_block.min()) / block_range
voxel_colors = plt.cm.viridis(scaled_values)

# Draw all selected array positions as filled cubes.
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")
ax.voxels(
    np.ones(small_block.shape, dtype=bool),
    facecolors=voxel_colors,
    edgecolor="black",
)
ax.set_xlabel("x position")
ax.set_ylabel("y position")
ax.set_zlabel("z position")
ax.set_title(f"A {small_block.shape} array drawn as voxels")
ax.set_box_aspect((1, 1, 1))
plt.show()


### Think together

Discuss these questions with a neighbor before moving on:

1. How would you change the code to make a `4 × 4 × 4` block instead?
2. In the plot, what do the colors represent?

<details>
<summary>Possible answers</summary>

1. One way is to change the stop value to `middle_x + 3`, `middle_y + 3`, and `middle_z + 3`. That includes four positions: `-1`, `0`, `+1`, and `+2` around the middle.
2. The colors represent the numbers stored inside the voxels. In this anatomical image, the numbers are image intensities: they tell the plotting program how bright or dark each voxel should be. They are **not** activation yet. Later, when we plot fMRI contrast maps, colors will mean something different: more signal for one task condition compared with another task condition.

</details>


## Brain slices and directions

Scientists view the 3D array as slices from three directions:

- **Sagittal** slices divide left and right. Nilearn calls this the `x` direction.
- **Coronal** slices divide front and back. Nilearn calls this the `y` direction.
- **Axial** slices divide top and bottom. Nilearn calls this the `z` direction.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/AnatomicalPlanes.png" alt="Sagittal, coronal, and axial brain planes" width="700">

## Plot the anatomical image

The default `ortho` view shows one sagittal, one coronal, and one axial slice together. The crosshairs mark the same location in all three views.

In [ ]:
plotting.plot_anat(
    anat_img,
    display_mode="ortho",
    title="Anatomical MRI: three slice directions",
)
plotting.show()

### Mini task: choose axial slice heights

Sometimes we want to compare several slices from the same direction. Here, `display_mode="z"` asks for **axial** slices, moving from the lower part of the brain to the top.

In the next code cell, you choose the slice heights in millimetres. Try to include low, middle, and high slices so you can see how the brain changes from bottom to top.

**Hint:** The hand motor area we will see later is fairly high in the brain, so include at least one slice around `z=50` or `z=70`.

These map coordinates are different from the array indices we used earlier. Nilearn reads the image information needed to place the voxels in brain space.


In [ ]:
# Mini task: choose the z-slice heights.
# Keep the numbers ordered from lower to higher.
# Include at least one high slice near the top of the brain.

slice_heights = [____, ____, ____, ____, ____, ____]

plotting.plot_anat(
    anat_img,
    display_mode="z",
    cut_coords=slice_heights,
    title="Axial slices from lower to higher positions",
)
plotting.show()


<details>
<summary>Emergency solution: click only if your group is stuck</summary>

```python
slice_heights = [-30, -10, 10, 30, 50, 70]
```

This set starts low in the brain and ends high near the top. The high slices help you get ready for the later hand-movement contrast.

</details>


## Explore the brain interactively

`view_img` creates a viewer you can click. Try clicking in each view and watch how the crosshair and the `x`, `y`, and `z` coordinates change.

In [ ]:
anat_viewer = plotting.view_img(
    anat_img,
    bg_img=False,
    cmap="gray",
    colorbar=False,
    symmetric_cmap=False,
    title="Click to explore the anatomical template",
)

anat_viewer

### Brain-region treasure hunt

 Use the interactive viewer to navigate near the following approximate landmark. The coordinates are clues, not exact borders: brain areas are larger than one point and differ between people.

**Top/parietal cortex:** move toward `(x=-40, y=-25, z=60)`. This is high on the side of the brain, near regions involved in body sensation and hand movement.

For each landmark, describe its location using ordinary words such as **left/right**, **front/back**, and **top/bottom**. Do not worry about finding one perfect voxel.


## Load an example fMRI scan

To explore real 4D data, we will load one small example scan supplied by Nilearn. The next cell downloads the files the first time it is run. We will first inspect the data shape and one voxel's time series. After that, we will look at what the participant was doing during the scan.

In [ ]:
# Download one participant's fMRI image and events table.
fmri_dataset = datasets.fetch_localizer_first_level()
fmri_img = image.load_img(fmri_dataset.epi_img)
events = pd.read_table(fmri_dataset.events)

print("fMRI image shape:", fmri_img.shape)
print("Number of dimensions:", len(fmri_img.shape))


## fMRI adds a fourth dimension: time

An anatomical image stores one value at every `(x, y, z)` voxel position. An fMRI image adds a fourth position, `t`, for **time**.

We can think about the same 4D data in two useful ways:

1. a sequence of 3D brain volumes, like frames in a movie, or
2. a time series from every voxel, showing how its signal changes during the scan.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/fmri_4d_timeseries.png" alt="A sequence of fMRI volumes and a time series from one voxel" width="700">

The **repetition time**, shortened to **TR**, is the time between one whole-brain volume and the next. In our example, the scanner records a new volume every 2.4 seconds.

The shape `(53, 63, 46, 128)` means:

- 53 voxel positions along the first spatial direction,
- 63 along the second,
- 46 along the third, and
- 128 time points.

The first three numbers describe one 3D volume. The fourth number tells us how many volumes were recorded over time.

### Extract one voxel's time series

The diagram above shows a line coming from one voxel. We can now create such a line from the real data.

To select one voxel, we choose one `x`, `y`, and `z` array position. The final `:` means **keep every time point**:

```python
fmri_data[x, y, z, :]
```

We will start with a voxel near the middle of the image.

In [ ]:
# Choose the middle array position in the fMRI image.
voxel_x = fmri_img.shape[0] // 2
voxel_y = fmri_img.shape[1] // 2
voxel_z = fmri_img.shape[2] // 2

# Read this voxel at every time point.
voxel_signal = np.asarray(
    fmri_img.dataobj[voxel_x, voxel_y, voxel_z, :]
)

print("Voxel index:", (voxel_x, voxel_y, voxel_z))
print("Time-series shape:", voxel_signal.shape)
print("First 10 signal values:", voxel_signal[:10])

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(voxel_signal, color="royalblue")
plt.xlabel("Time point (volume number)")
plt.ylabel("fMRI signal intensity")
plt.title("Signal over time from one voxel")
plt.grid(alpha=0.25)
plt.show()

### What is the fMRI signal?

At every time point, the scanner stores a signal-intensity number for every voxel. Part of this number is influenced by the **BOLD signal**. BOLD stands for **blood-oxygen-level dependent**.

When a group of brain cells becomes more active, it needs energy. The body changes the local supply of oxygen-rich blood. fMRI is sensitive to these blood-oxygen changes. It does **not** measure thoughts or electrical brain activity directly.

A voxel's line can also change because of breathing, heartbeat, head movement, scanner noise, and slow changes during the scan. Therefore, one bump in one time series is not enough to claim that a task activated that voxel.

## Look at one fMRI volume

The time series used one voxel across all 128 time points. We can also do the opposite: choose one time point and look at every voxel in that 3D volume.

`index_img(fmri_img, 0)` selects the first volume. It looks blurrier and less detailed than the anatomical image because functional images are collected quickly, over and over again.

In [ ]:
first_volume = image.index_img(fmri_img, 0)

plotting.plot_epi(
    first_volume,
    title="The first 3D volume in the fMRI scan",
)
plotting.show()

### What do the colors mean?

In this image, brighter and darker colors show the **raw signal intensity** recorded in the voxels at one moment.

These colors do **not** yet show task activation. A bright voxel is not automatically a voxel that was busy during a button press or a sentence. Its intensity also depends on tissue type, scanner sensitivity, image position, and other sources of variation.

To say something about a task, we need to connect the changing voxel signal to the experiment and compare **one condition against another condition**.

## The localizer experiment

To compare task conditions, we will use data from one participant in the localizer study by Pinel and colleagues. During a short scan, the participant saw or heard many brief instructions and stimuli.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/pinel_localizer_tasks.png" alt="Examples of visual and audio tasks in the Pinel localizer experiment" width="850">

The tasks included:

- reading short sentences,
- listening to short sentences,
- doing simple subtraction by sight or sound,
- pressing the left or right button after a visual or spoken instruction, and
- looking at flashing checkerboards.

If you have never seen an fMRI study before, these conditions might look a bit arbitrary. But they are chosen carefully. Each condition helps researchers ask a simple question about an important brain function.

For example, checkerboards help us ask about **vision**: how does the brain react to things we see? Button presses help us ask about **movement**: how does the brain send commands to move our hands? Sentences help us ask about **language**: how does the brain understand speech or written words? Subtraction tasks help us ask about **numbers**: how does the brain work with simple arithmetic?

Because the experiment contains several task types, we can compare them in different ways. For example: Which areas respond more to horizontal than vertical checkerboards? How does listening differ from reading? How do right- and left-hand button presses differ?

The data set also contains an **events table** recording what happened and when. That table lets us connect the task conditions to the fMRI signal.


## Connect the scan to the experiment

The scanner only records changing numbers. It does not know whether the participant was reading, listening, or pressing a button.

The **events table** supplies that missing information. Each row describes one event:

- `trial_type`: what the participant did,
- `onset`: when it began, in seconds after the scan started,
- `duration`: how long it lasted, in seconds.

The analysis uses these timings to line up the experiment with the 128 measured brain volumes.

In [ ]:
# Display the first ten events in the experiment.
events.head(10)

## Fit the model

Now we want to connect two things:

1. the fMRI signal from each voxel, and
2. the timing of the task conditions.

For each condition, the computer builds a simple **predictor**. A predictor is like a clue that says: "If this task matters for this voxel, the signal should change around these times."

A useful analogy is mixing sound sliders. Imagine one slider for right-hand button presses, one slider for left-hand button presses, one for reading, one for listening, and so on. For each voxel, the model asks: which sliders need to be turned up to explain this voxel's signal?

The model gives every predictor a number called a **weight**. We do not need the math here. The important idea is:

- a larger weight means this task timing was more useful for explaining that voxel,
- a weight near zero means this task timing did not help much,
- the model repeats this for every voxel in the brain.

The brain response is slow, so Nilearn also knows that the fMRI signal usually rises a few seconds after a task event instead of instantly. We will let Nilearn handle that detail for us.

Our goal is simple: find out where the **right-hand predictor** was more important than the **left-hand predictor**.


In [ ]:
# Describe and fit one participant's first-level model.
first_level_model = FirstLevelModel(
    t_r=2.4,
    slice_time_ref=0,
    hrf_model="spm",
    drift_model="cosine",
    high_pass=0.01,
    smoothing_fwhm=4,
    minimize_memory=False,
)

first_level_model = first_level_model.fit(fmri_img, events=events)
print("Model fitted successfully.")

## A contrast: right hand compared with left hand

After fitting, every voxel has a model weight for every condition. A **contrast** compares those weights.

Our question is:

> Where was the modeled response stronger for right-hand button presses than for left-hand button presses?

In short:

```text
average right-hand weight - average left-hand weight
```

Subtracting matters. A colorful map of one condition alone cannot tell us whether the response is especially strong for that condition. A contrast makes the comparison explicit.

In [ ]:
# Average across visual and audio instructions for each hand.
right_minus_left = (
    "0.5 * (visual_right_hand_button_press "
    "+ audio_right_hand_button_press) "
    "- 0.5 * (visual_left_hand_button_press "
    "+ audio_left_hand_button_press)"
)

right_minus_left_map = first_level_model.compute_contrast(
    right_minus_left,
    output_type="z_score",
)

print("Contrast calculated.")

### Plot the contrast map

The plot below visualizes the contrast:


In [ ]:
# Find the strongest right-hand-greater voxel and convert it to map coordinates.
contrast_data = right_minus_left_map.get_fdata()
peak_index = np.unravel_index(np.nanargmax(contrast_data), contrast_data.shape)
peak_coord = right_minus_left_map.affine @ [*peak_index, 1]
peak_coord = peak_coord[:3]

plotting.plot_stat_map(
    right_minus_left_map,
    bg_img=anat_img,
    display_mode="ortho",
    cut_coords=peak_coord,
    threshold=3.0,
    symmetric_cbar=True,
    title="Focused view: strongest right-hand-greater cluster",
)
plotting.show()

plotting.plot_stat_map(
    right_minus_left_map,
    bg_img=anat_img,
    display_mode="z",
    cut_coords=[40, 50, 60, 70],
    threshold=3.0,
    symmetric_cbar=True,
    title="Right-hand button press minus left-hand button press",
)
plotting.show()


Look carefully at the sides of the brain:

1. Where in the brain do we observe the strongest responses? Top or bottom?
2. Where are the strongest warm colors? Are they more on the left or right side of the brain?
3. Where are the strongest cool colors? Are they more on the left or right side of the brain?

### What did we just uncover?

The contrast map shows us something important about how the brain controls the body.

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/Anatomytool_Sensory_homunculus_English.jpg" alt="Motor homunculus showing body-part representation along the motor cortex" width="650">

This picture is called a **motor homunculus**. It is a simplified map of the motor cortex. Different parts of this strip of brain tissue are linked to movements of different body parts. The hand has a large area because hands can make many precise movements.

In our contrast, we asked:

```text
right-hand button press - left-hand button press
```

That means the colors should be read like this:

- **Warm colors** mean: more modeled activity for the **right hand** than the left hand.
- **Cool colors** mean: less modeled activity for the **right hand** than the left hand. In other words, these voxels responded more for the **left hand**.

We just uncovered an important principle of brain organization: **contralaterality**. This means that one side of the brain is mainly connected to the opposite side of the body. The left motor cortex is strongly involved in moving the right hand. The right motor cortex is strongly involved in moving the left hand.

This kind of crossing appears in other brain systems too. For example, the left side of the visual world is processed mostly by the right side of the brain, and the right side of the visual world is processed mostly by the left side. Hearing is a bit more mixed, because both ears send information to both sides of the brain, but the opposite side is often especially important.

So the result is not just about button presses. It is one example of a larger idea: the brain is organized in maps, and many of those maps cross from one side of the body or world to the other side of the brain.


## Sources and further reading

This notebook adapts teaching material and figures from `TEWA2/02_Understanding_MRI_Data.ipynb` and `TEWA2/06_First_Level_Analysis.ipynb` in this repository.

- [Nilearn: 3D and 4D images](https://nilearn.github.io/stable/auto_examples/00_tutorials/plot_3d_and_4d_niimg.html)
- [Nilearn: first-level models](https://nilearn.github.io/stable/glm/first_level_model.html)
- [Nilearn plotting tools](https://nilearn.github.io/stable/modules/plotting.html)
- Pinel, P., Thirion, B., Meriaux, S. et al. (2007). Fast reproducible identification and large-scale databasing of individual functional cognitive networks. *BMC Neuroscience, 8*, 91. https://doi.org/10.1186/1471-2202-8-91